# Application of Radial Equilibrium Equation for a Rotor - Inverse Problem
## Adaptation of the code RE-DES by Lewis (Turbomachines Performance Analysis)

The radial equilibrium equation can be reordered and written as

$$\boxed{
\frac{\text{d}}{\text{d}r} c_x(r)^2 = 2\left( \omega - \frac{c_\theta(r)}{r}\right)
\frac{\text{d} (rc_\theta(r))}{\text{d}r}
}  \tag{1}
$$

that gives the solution
$$
c_x(r) = \sqrt{f(r) + k} \tag{2}
$$
where
$$
f(r) = 2 \int_{r_h}^r \left( \omega - \frac{c_\theta(r)}{r}\right)\text{d} (rc_\theta(r)) \tag{3}
$$

The aim is, given all the data: $Q$, $\omega$, $r_h$, $r_t$ and the function
$c_\theta(r)$, compute $c_x(r)$ and the angle $\beta_2(r)$ that fullfils the
required flowrate.

The necessary modules are imported

In [1]:
import numpy as np
import pandas as pd
pd.set_option('display.precision', 3)
from scipy.interpolate import CubicSpline
from scipy.integrate import cumulative_trapezoid, trapezoid

The data for the fan is input in this cell.
The hub to tip ratio and the rms radius, $r_\text{rms} = \sqrt{\frac{r_h^2+r_t^2}{2}}$ are computed, and the swirl distribution is also chosen.
For numerical computations, the $r$ domain is divided in $m$ intervals, and $n$ for output of results. Velocity at $r_{rms}$, $c_{\theta,\text{rms}}=c_\theta(r=r_\text{rms})$, is calculated from Euler's equation, and $c_x$ in $r_\text{rms}$ is as well estimated, assuming that it is the average velocity given by the flow rate, $\overline{c_x}$.

In [2]:
rho = 1.2                           # Density of air, kg/m³
Dh = 0.064                          # Diameter of hub, m
rh = Dh/2                           # Radius of hub, m
Dt = 0.25                           # Diameter of tip, m
rt = Dt/2                           # Radius of tip, m
h = rh/rt                           # Hub-tip ratio
rrms = np.sqrt(0.5*(rh*rh+rt*rt))   # RMS radius, m
n = 10                              # number of outputs
m = 601                             # number of interpolation points
r = np.linspace(rh,rt,m)            # Discretization of the radius for interpolation, m
Qdata = 1716                        # Flow rate, m³/h
Qdata = Qdata/3600                  # Flow rate, m³/s
omega = 2275                        # Rotational speed, rpm
omega = omega*np.pi/30              # Rotational speed, rad/s
Delta_p0_target = 70                # Pressure rise, Pa

ctrms = Delta_p0_target/(rho*omega*rrms)         # c_theta,rms, m/s
cxm = Qdata/(np.pi*(rt*rt-rh*rh))   # c_x,rms, m/s
print("Flow rate = {:0.4f} m³/s".format(Qdata))
print("Hub to tip ratio = {:0.4f}".format(h))
print("rms = {:.4f} m".format(rrms))
print("ctheta_rms = {:.4f} m/s".format(ctrms))
print("cx_rms = {:.4f} m/s".format(cxm))


Flow rate = 0.4767 m³/s
Hub to tip ratio = 0.2560
rms = 0.0912 m
ctheta_rms = 2.6837 m/s
cx_rms = 10.3916 m/s


In the next cell the type of the angular flow is chosen. These are the options: 

- **Free Vortex**:

  $$ c_\theta = \frac{B}{r} $$
  where
  $$ B = c_{\theta,rms}r_{rms} $$
- **Forced Vortex**:
  
  $$ c_\theta = Ar $$
  where 
  $$ A = \frac{c_{\theta,rms}}{r_{rms}} $$
- **Constant Vortex**:
  
  $$ c_\theta = c_{\theta,rms} $$
- **Mixed Vortex**:
  
  $$ c_\theta = A(r-r_{rms}) + \frac{B}{r} $$
  where
  $$ B = c_{\theta,rms}r_{rms} $$
  and
  $$ A = \frac{\Delta c_\theta}{r_t - r_h} + \frac{B}{r_t r_h} $$
  where $\Delta c_\theta$ is the variability of $c_\theta$ between the tip and the hub, that is, some kind of "strength" of the vortex
- **Arbitray Vortex**:
  
  The user can define any function $c_\theta(r)$
- **Table**:
  
  Tangential velocity can be given as an array, along with the $r_i$ points


In [3]:
# User choice: "free", "forced", "constant", "mixed", "arbitrary", "table"
flow_type = "mixed"

if flow_type == "free":
    B = ctrms*rrms
    ctheta = B/r
    print("Free vortex flow")
    print("B = {:.4f} m²/s".format(B))
elif flow_type == "forced":
    A = ctrms/(rrms)
    ctheta = A*r
    print("Forced vortex flow")
    print("A = {:.4f} 1/s".format(A))
elif flow_type == "constant":
    ctheta = ctrms*np.ones(r.size)
    print("Constant ctheta flow")
    print("ctheta = {:.4f} m/s".format(ctrms))
elif flow_type == "mixed":
    delta_ctheta = 1
    B = ctrms*rrms
    A = delta_ctheta/(rt-rh) + B/(rt*rh)
    ctheta = A*(r-rrms) + B/r
    print("Mixed vortex flow")
    print("A = {:.4f} 1/s".format(A))
    print("B = {:.4f} m²/s".format(B))
elif flow_type == "arbitrary":
    ctheta = 2*omega/3*r-9.08/np.sqrt(r)
elif flow_type == "table":
    # For example, values obtained from a reference design, CFD, or experimental extraction
    c_theta_array = np.array([0.90278909, 1.28974431, 1.66424824, 2.03205183, 2.39580352,
       2.75690324, 3.11616483, 3.47409568, 3.83102958, 4.1871957 ])
    r_stations = np.linspace(rh, rt, len(c_theta_array))
    ctheta_spline = CubicSpline(r_stations, c_theta_array)
    ctheta = ctheta_spline(r)

Mixed vortex flow
A = 71.9661 1/s
B = 0.2449 m²/s


And now, the function $f(r)$ is computed by numerical integration (Eq. (3))

In [4]:
f = 2.0 * cumulative_trapezoid(omega-np.divide(ctheta,r),
                         np.multiply(r,ctheta),initial=0)


### First approximation

The first approximation of the value of $k$ is made with the assumption that
$c_{x,\text{rms}} = \overline{c_x}$

In [5]:
frms = CubicSpline(r,f)(rrms)
k = cxm*cxm-frms
print("First approximation of k: \n k = {:.4f} m²/s²".format(k))

First approximation of k: 
 k = 49.6024 m²/s²


Values of $DF < 0.6$ and a first estimation of $C_D$ are defined. With this assumptions and data, ${C_L}$, ${σ}$ (solidity) and ${c_x}$ along the entire length of the profile are calculated.

With these data, the chord of the profile along the length of the blade is calculated, thus defining the geometry of the blade.


$$
σ = \frac {cos(β_1) (tan(β_1) - tan(β_2))}{2 D_F - 2  [1 - \frac{cos(β_1)}{cos(β_2) } ] } \tag{4}
$$

$$
C_L = \frac {2  cos(β_m)  (tan(β_1) - tan(β_2))}{σ} - C_Dtan(β_m) \tag{5}
$$

The contribution of Samuel Limonchi (course 2023-24 of MUREM) to this part of the notebook is acknowledged.

In [6]:
DF_target = 0.3
CD = 0.05
nu = 1.5e-5 # Kinematic viscosity of air, m²/s
Nblades = 5
def solve_fan(k):
    cx = np.sqrt(np.maximum(k + f, 1.0e-6))
    Q = 2*np.pi*trapezoid(np.multiply(r,cx),r)
    alpha2 = np.arctan(np.divide(ctheta,cx))
    beta2 = np.arctan(np.divide(omega*r-ctheta,cx))
    beta1 = np.arctan(np.divide(omega*r,cx))
    cosbeta1 = np.cos(beta1)
    cosbeta2 = np.cos(beta2)
    tanbeta1 = np.tan(beta1)
    tanbeta2 = np.tan(beta2)
    tanbetam = 0.5*(tanbeta1 + tanbeta2)
    beta_m = np.arctan(tanbetam)
    cosbetam = 1 / np.sqrt(1 + tanbetam*tanbetam)
    solidity = (cosbeta1* (tanbeta1 - tanbeta2)) / (2*DF_target - 2 * (1 - (cosbeta1/cosbeta2) ))
    bladespace = 2*np.pi*r/Nblades
    chord =  solidity * bladespace
    CL = (2/solidity) * (cosbetam * (tanbeta1 - tanbeta2)) - CD*tanbetam
    Delta_p0 = rho * omega * r * ctheta
    Delta_p0_avg = 2*trapezoid(np.multiply(r,Delta_p0),r)/(rt*rt-rh*rh)
    ## Computation of total pressure loss and net total pressure rise
    Wm = cx / cosbetam
    Delta_p0_loss = 0.5 * rho * Wm * Wm * solidity * CD / cosbetam
    Delta_p0_net = Delta_p0 - Delta_p0_loss
    efficiency = Delta_p0_net / Delta_p0
    Delta_p0_avg_net = 2*trapezoid(np.multiply(r,Delta_p0_net),r)/(rt*rt-rh*rh)
    efficiency_avg = Delta_p0_avg_net / Delta_p0_avg
    Re = np.divide(np.multiply(Wm, chord), nu)
    #
    print("Q = {:.3f} m^3/s".format(Q))
    print("Delta p0 avg = {:.2f} Pa".format(Delta_p0_avg))
    print("Delta p0 avg net = {:.2f} Pa".format(Delta_p0_avg_net))
    print("efficiency avg = {:.2f} %".format(efficiency_avg*100))
    errorQ = np.abs(Qdata-Q)/Qdata * 100
    print("error in flow rate = {:.1e} %".format(errorQ))
    errorP0 = np.abs(Delta_p0_target-Delta_p0_avg_net)/Delta_p0_target * 100
    print("error in pressure rise = {:.1e} %".format(errorP0))
    #
    # Results are interpolated to a smaller number of points for output
    #
    rdata = np.linspace(rh, rt, n)
    ctheta_data = CubicSpline(r,ctheta)(rdata)
    cx_data = CubicSpline(r,cx)(rdata)
    alpha2_data = np.rad2deg(CubicSpline(r,alpha2)(rdata))
    beta2_data = np.rad2deg(CubicSpline(r,beta2)(rdata))
    beta1_data = np.rad2deg(CubicSpline(r,beta1)(rdata))
    beta_m_data = np.rad2deg(CubicSpline(r,beta_m)(rdata))
    solidity_data = CubicSpline(r,solidity)(rdata)
    Wm_data = CubicSpline(r,Wm)(rdata)
    CL_data = CubicSpline(r,CL)(rdata)
    Delta_p0_data = CubicSpline(r,Delta_p0)(rdata)
    Delta_p0_loss_data = CubicSpline(r,Delta_p0_loss)(rdata)
    Delta_p0_net_data = CubicSpline(r,Delta_p0_net)(rdata)
    efficiency_data = CubicSpline(r,efficiency)(rdata)
    bladespace_data = 2*np.pi*rdata/Nblades
    chord_data =  solidity_data * bladespace_data
    Re_data = CubicSpline(r,Re)(rdata)
    #
    # Pandas DataFrame is created for output
    #
    df = pd.DataFrame({
        "Radius (m)": rdata,
        "c_theta (m/s)": ctheta_data,
        "c_x (m/s)": cx_data,
        "alpha_2 (deg)": alpha2_data,
        "beta_1 (deg)": beta1_data,
        "beta_2 (deg)": beta2_data,
        "beta_m (deg)": beta_m_data,
        "Solidity": solidity_data,
        "CL": CL_data,
        "Chord (mm)": chord_data * 1000,
        "Wm (m/s)": Wm_data,
        "Delta p0_Euler (Pa)": Delta_p0_data,
        "Delta_p0_loss (Pa)": Delta_p0_loss_data,
        "Delta_p0_net (Pa)": Delta_p0_net_data,
        "efficiency": efficiency_data,
        "Reynolds number": Re_data
    })
    df = df.round({"Chord (mm)":0, "Reynolds number":0})
    return df

In [7]:

df = solve_fan(k)

Q = 0.481 m^3/s
Delta p0 avg = 76.68 Pa
Delta p0 avg net = 66.10 Pa
efficiency avg = 86.20 %
error in flow rate = 8.5e-01 %
error in pressure rise = 5.6e+00 %


In [8]:
df

,Radius (m),c_theta (m/s),c_x (m/s),alpha_2 (deg),beta_1 (deg),beta_2 (deg),beta_m (deg),Solidity,CL,Chord (mm),Wm (m/s),Delta p0_Euler (Pa),Delta_p0_loss (Pa),Delta_p0_net (Pa),efficiency,Reynolds number
0,0.032,3.389,7.043,25.693,47.267,31.020,40.094,1.778,0.372,71.0,9.206,30.999,5.910,25.089,0.809,43881.0
1,0.042,2.264,6.755,18.533,56.188,49.184,52.967,0.616,0.589,33.0,11.215,27.405,3.861,23.544,0.859,24512.0
2,0.053,1.873,6.838,15.319,61.409,57.354,59.503,0.350,0.709,23.0,13.475,28.205,3.760,24.445,0.867,20826.0
3,0.063,1.854,7.367,14.128,63.856,60.749,62.383,0.275,0.753,22.0,15.893,33.398,4.493,28.906,0.865,23054.0
4,0.073,2.050,8.267,13.929,64.676,61.803,63.311,0.258,0.763,24.0,18.407,42.985,5.848,37.137,0.864,29222.0
5,0.084,2.382,9.429,14.176,64.685,61.755,63.294,0.265,0.756,28.0,20.980,56.966,7.796,49.170,0.863,39016.0
6,0.094,2.804,10.760,14.604,64.337,61.223,62.862,0.283,0.743,33.0,23.590,75.340,10.344,64.996,0.863,52505.0
7,0.104,3.289,12.200,15.089,63.857,60.504,62.274,0.305,0.728,40.0,26.222,98.108,13.513,84.596,0.862,69852.0
8,0.115,3.821,13.709,15.576,63.351,59.738,61.650,0.329,0.712,47.0,28.870,125.270,17.322,107.948,0.862,91234.0
9,0.125,4.389,15.264,16.041,62.863,58.988,61.043,0.354,0.696,56.0,31.527,156.826,21.792,135.034,0.861,116817.0


### More precise computation


Instead of estimating $k$ with the assumption of $c_x$ in $r_{rms}$ being the average value, a more accurate computation
can be performed forcing the flow rate to be the input one (equation (5.47) and figure 5.6)

In [9]:
from scipy.optimize import brentq

def QFunction(k):
    cx = np.sqrt(np.maximum(k + f, 1.0e-6))
    Q_temptative = 2*np.pi*trapezoid(np.multiply(r,cx),r)
    return Qdata-Q_temptative

k = brentq(QFunction,min(0.5*k,5*k),max(0.5*k,5*k))
print("k = {:.4f} m²/s²".format(k))

k = 47.8954 m²/s²


In [10]:
df = solve_fan(k)
df

Q = 0.477 m^3/s
Delta p0 avg = 76.68 Pa
Delta p0 avg net = 66.03 Pa
efficiency avg = 86.12 %
error in flow rate = 4.7e-14 %
error in pressure rise = 5.7e+00 %


,Radius (m),c_theta (m/s),c_x (m/s),alpha_2 (deg),beta_1 (deg),beta_2 (deg),beta_m (deg),Solidity,CL,Chord (mm),Wm (m/s),Delta p0_Euler (Pa),Delta_p0_loss (Pa),Delta_p0_net (Pa),efficiency,Reynolds number
0,0.032,3.389,6.921,26.087,47.767,31.465,40.589,1.870,0.355,75.0,9.113,30.999,6.134,24.865,0.802,45676.0
1,0.042,2.264,6.627,18.865,56.691,49.723,53.491,0.628,0.580,33.0,11.139,27.405,3.927,23.478,0.857,24798.0
2,0.053,1.873,6.712,15.593,61.854,57.836,59.966,0.354,0.703,23.0,13.411,28.205,3.813,24.392,0.865,20928.0
3,0.063,1.854,7.250,14.346,64.216,61.138,62.757,0.277,0.750,22.0,15.839,33.398,4.547,28.851,0.864,23119.0
4,0.073,2.050,8.163,14.099,64.955,62.104,63.601,0.260,0.760,24.0,18.360,42.985,5.904,37.081,0.863,29281.0
5,0.084,2.382,9.338,14.308,64.899,61.986,63.516,0.266,0.754,28.0,20.939,56.966,7.853,49.112,0.862,39078.0
6,0.094,2.804,10.680,14.708,64.502,61.402,63.034,0.283,0.742,33.0,23.553,75.340,10.403,64.937,0.862,52574.0
7,0.104,3.289,12.130,15.172,63.988,60.646,62.410,0.305,0.727,40.0,26.190,98.108,13.573,84.536,0.862,69932.0
8,0.115,3.821,13.647,15.643,63.455,59.852,61.759,0.330,0.711,47.0,28.840,125.270,17.383,107.887,0.861,91327.0
9,0.125,4.389,15.208,16.097,62.948,59.081,61.133,0.354,0.695,56.0,31.500,156.826,21.855,134.971,0.861,116923.0


In [11]:
df['beta_2 (deg)'].to_numpy()

array([31.46458518, 49.72333857, 57.8357223 , 61.13792921, 62.10363937,
       61.98550721, 61.40175539, 60.64550684, 59.85194521, 59.08138468])